In [1]:
import pandas as pd
import numpy as np

In [2]:
df= pd.read_csv("/content/Business Retail.csv")


In [3]:
df.head()

,Order_ID,Customer_ID,Order_Date,Region,Product_Category,Customer_Segment,Quantity,Unit_Price,Discount_Rate,Revenue,Cost,Profit,Payment_Method
0,1,CUST3818,8/18/2024,North,Clothing,Corporate,5,300.68,0.27,1097.48,768.29,329.19,Credit Card
1,2,CUST9689,6/19/2024,South,Beauty,Home Office,9,32.89,0.02,290.09,179.33,110.76,Debit Card
2,3,CUST9147,11/21/2024,West,Sports,Corporate,5,345.61,0.25,1296.04,1022.60,273.44,Credit Card
3,4,CUST7938,7/19/2024,North,Clothing,Consumer,1,444.50,0.06,417.83,280.99,136.84,UPI
4,5,CUST5127,10/28/2024,South,Home & Kitchen,Consumer,5,65.13,0.21,257.26,151.90,105.36,Credit Card


In [4]:
df.info()
df.describe()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Order_ID          10000 non-null  int64  
 1   Customer_ID       10000 non-null  object 
 2   Order_Date        10000 non-null  object 
 3   Region            10000 non-null  object 
 4   Product_Category  10000 non-null  object 
 5   Customer_Segment  10000 non-null  object 
 6   Quantity          10000 non-null  int64  
 7   Unit_Price        10000 non-null  float64
 8   Discount_Rate     10000 non-null  float64
 9   Revenue           10000 non-null  float64
 10  Cost              10000 non-null  float64
 11  Profit            10000 non-null  float64
 12  Payment_Method    10000 non-null  object 
dtypes: float64(5), int64(2), object(6)
memory usage: 1015.8+ KB


,0
Order_ID,0
Customer_ID,0
Order_Date,0
Region,0
Product_Category,0
Customer_Segment,0
Quantity,0
Unit_Price,0
Discount_Rate,0
Revenue,0


In [5]:

df['Order_Date'] = pd.to_datetime(df['Order_Date'])

numeric_cols = ['Quantity', 'Unit_Price', 'Discount_Rate',
                'Revenue', 'Cost', 'Profit']

df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')

In [6]:
df = df.drop_duplicates(subset='Order_ID')

In [7]:
df = df.dropna(subset=['Revenue', 'Cost', 'Profit'])

In [8]:
df['Discount_Rate'] = df['Discount_Rate'].fillna(0)

In [10]:
df['Calculated_Revenue'] = (
    df['Quantity'] * df['Unit_Price'] * (1 - df['Discount_Rate'])
)

In [11]:
df['Revenue_Difference'] = df['Revenue'] - df['Calculated_Revenue']

df['Revenue_Difference'].describe()

,Revenue_Difference
count,10000.000000
mean,-0.000020
std,0.002839
min,-0.005000
25%,-0.002400
50%,0.000000
75%,0.002400
max,0.005000


In [12]:
df['Revenue'] = df['Calculated_Revenue']

In [13]:
df['Profit'] = df['Revenue'] - df['Cost']


In [14]:
df = df[df['Quantity'] > 0]
df = df[df['Unit_Price'] > 0]
df = df[df['Cost'] >= 0]


In [15]:
Q1 = df['Revenue'].quantile(0.25)
Q3 = df['Revenue'].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

df = df[(df['Revenue'] >= lower) & (df['Revenue'] <= upper)]


In [16]:
df['Year'] = df['Order_Date'].dt.year
df['Month'] = df['Order_Date'].dt.month
df['Month_Name'] = df['Order_Date'].dt.month_name()
df['Year_Month'] = df['Order_Date'].dt.to_period('M')


In [17]:
def discount_category(x):
    if x == 0:
        return "No Discount"
    elif x <= 0.1:
        return "Low"
    elif x <= 0.3:
        return "Medium"
    else:
        return "High"

df['Discount_Category'] = df['Discount_Rate'].apply(discount_category)


In [18]:
text_cols = ['Region', 'Product_Category',
             'Customer_Segment', 'Payment_Method']

for col in text_cols:
    df[col] = df[col].str.strip().str.title()


In [19]:
df.info()
df.isnull().sum()


<class 'pandas.core.frame.DataFrame'>
Index: 9847 entries, 0 to 9999
Data columns (total 20 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Order_ID            9847 non-null   int64         
 1   Customer_ID         9847 non-null   object        
 2   Order_Date          9847 non-null   datetime64[ns]
 3   Region              9847 non-null   object        
 4   Product_Category    9847 non-null   object        
 5   Customer_Segment    9847 non-null   object        
 6   Quantity            9847 non-null   int64         
 7   Unit_Price          9847 non-null   float64       
 8   Discount_Rate       9847 non-null   float64       
 9   Revenue             9847 non-null   float64       
 10  Cost                9847 non-null   float64       
 11  Profit              9847 non-null   float64       
 12  Payment_Method      9847 non-null   object        
 13  Calculated_Revenue  9847 non-null   float64       
 1

,0
Order_ID,0
Customer_ID,0
Order_Date,0
Region,0
Product_Category,0
Customer_Segment,0
Quantity,0
Unit_Price,0
Discount_Rate,0
Revenue,0


In [20]:
df.to_csv("Cleaned_Business_Analytics_Dataset.csv", index=False)


In [21]:
df.head()

,Order_ID,Customer_ID,Order_Date,Region,Product_Category,Customer_Segment,Quantity,Unit_Price,Discount_Rate,Revenue,Cost,Profit,Payment_Method,Calculated_Revenue,Revenue_Difference,Year,Month,Month_Name,Year_Month,Discount_Category
0,1,CUST3818,2024-08-18,North,Clothing,Corporate,5,300.68,0.27,1097.4820,768.29,329.1920,Credit Card,1097.4820,-0.0020,2024,8,August,2024-08,Medium
1,2,CUST9689,2024-06-19,South,Beauty,Home Office,9,32.89,0.02,290.0898,179.33,110.7598,Debit Card,290.0898,0.0002,2024,6,June,2024-06,Low
2,3,CUST9147,2024-11-21,West,Sports,Corporate,5,345.61,0.25,1296.0375,1022.60,273.4375,Credit Card,1296.0375,0.0025,2024,11,November,2024-11,Medium
3,4,CUST7938,2024-07-19,North,Clothing,Consumer,1,444.50,0.06,417.8300,280.99,136.8400,Upi,417.8300,0.0000,2024,7,July,2024-07,Low
4,5,CUST5127,2024-10-28,South,Home & Kitchen,Consumer,5,65.13,0.21,257.2635,151.90,105.3635,Credit Card,257.2635,-0.0035,2024,10,October,2024-10,Medium


In [22]:
from google.colab import files
files.download("Cleaned_Business_Analytics_Dataset.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>